<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 08 — Aprendizaje supervisado: Regresión sobre la vista minable municipal

**Pregunta:** ¿podemos predecir el **puntaje global promedio** de un municipio-año
a partir de su perfil de conectividad, pobreza y educación?

**Modelos** (los 2 cubiertos en clase para regresión):

1. `LinearRegression` — modelo lineal regularizado (Ridge/ElasticNet)
2. `DecisionTreeRegressor` — árbol de regresión

**Vista minable:** `gold/panel_municipal` (joined de ICFES ⋈ Internet ⋈ SISBEN ⋈ MEN,
grano municipio-año, 4 fuentes unidas → un solo dataset).

**Pipeline de la clase (sección 1.5):**
`fillna(median) → VectorAssembler → StandardScaler → Regressor`

## 1. Setup y carga de la vista minable

In [1]:
import sys
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = get_spark("Entrega2-ML-Regresion", executor_memory="4g", driver_memory="3g", cores=2)

panel = spark.read.parquet(P.GOLD_PANEL_MUNICIPAL)
TARGET = "avg_punt_global"

# Filtro de calidad: solo filas con target válido y tamaño mínimo
panel_ml = (panel
    .filter(col(TARGET).isNotNull())
    .filter(col("n_estudiantes") >= 30))
print(f"Filas de la vista minable: {panel.count():,}")
print(f"Filas para ML (con target y n>=30): {panel_ml.count():,}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/22 08:20:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Filas de la vista minable: 7,004


Filas para ML (con target y n>=30): 6,502


## 2. Selección inicial de features

In [2]:
FEATURES = [
    "pct_internet_icfes",          # conectividad — declarada por estudiante
    "accesos_per_capita_5_16",     # conectividad — MinTIC normalizado por población
    "avg_velocidad_bajada",        # calidad de internet
    "n_proveedores",               # diversidad de oferta
    "idx_privacion",               # pobreza — suma de 15 indicadores SISBEN
    "pct_grupo_A",                 # pobreza — % en grupo más vulnerable
    "pct_rural_sisben",            # ruralidad
    "COBERTURA_NETA",              # educación — oferta
    "DESERCION",                   # educación — calidad
    "APROBACION",                  # educación — calidad
    "n_estudiantes",               # tamaño
    "POBLACION_5_16",              # demografía
]
# Cast a double (algunas vienen long o decimal)
for f in FEATURES:
    panel_ml = panel_ml.withColumn(f, col(f).cast("double"))
print(f"Features candidatas: {len(FEATURES)}")

Features candidatas: 12


## 3. Reporte de nulos y filtrado de features 100% nulas

In [3]:
from pyspark.sql.functions import count, when

null_counts = (panel_ml
    .select([F.sum(col(c).isNull().cast("int")).alias(c) for c in FEATURES])
    .first().asDict())
total = panel_ml.count()

print(f"{'feature':<28s} {'nulls':>8s} {'%null':>8s}")
USABLE = []
for c in FEATURES:
    n = null_counts[c]
    pct = 100*n/total
    flag = "DESCARTAR" if n == total else "ok"
    print(f"  {c:<28s} {n:>8d} {pct:>7.1f}%  {flag}")
    if n < total:
        USABLE.append(c)
print(f"\nFeatures usables: {len(USABLE)} / {len(FEATURES)}")

feature                         nulls    %null
  pct_internet_icfes                  0     0.0%  ok
  accesos_per_capita_5_16          3120    48.0%  ok
  avg_velocidad_bajada             3119    48.0%  ok
  n_proveedores                    3119    48.0%  ok
  idx_privacion                      41     0.6%  ok
  pct_grupo_A                        41     0.6%  ok
  pct_rural_sisben                   41     0.6%  ok
  COBERTURA_NETA                     23     0.4%  ok
  DESERCION                          13     0.2%  ok
  APROBACION                          5     0.1%  ok
  n_estudiantes                       0     0.0%  ok
  POBLACION_5_16                      5     0.1%  ok

Features usables: 12 / 12


## 4. Imputación de nulos (mediana — patrón de clase, sección 1.3)

In [4]:
# Para cada feature numérica con nulos, computamos la mediana con approxQuantile
# (igual a la clase) y aplicamos fillna.
medianas = {}
for c in USABLE:
    q = panel_ml.approxQuantile(c, [0.5], 0.01)
    if q:
        medianas[c] = q[0]
print("Medianas calculadas:")
for c, m in medianas.items():
    print(f"  {c:<28s} = {m:.4f}")

panel_imp = panel_ml.fillna(medianas)
# Verificación
nulos_post = panel_imp.select(
    [F.sum(col(c).isNull().cast("int")).alias(c) for c in USABLE]
).first().asDict()
print(f"\nNulos tras imputación: {sum(nulos_post.values())}")

Medianas calculadas:
  pct_internet_icfes           = 0.2398
  accesos_per_capita_5_16      = 0.4425
  avg_velocidad_bajada         = 19.2700
  n_proveedores                = 7.0000
  idx_privacion                = 3.7637
  pct_grupo_A                  = 0.3742
  pct_rural_sisben             = 0.5492
  COBERTURA_NETA               = 0.8628
  DESERCION                    = 0.0306
  APROBACION                   = 0.9208
  n_estudiantes                = 175.0000
  POBLACION_5_16               = 3228.0000



Nulos tras imputación: 0


## 5. Análisis de correlación (df.stat.corr — patrón clase) y eliminación de redundantes

In [5]:
# Matriz de Pearson entre todas las features e incluyendo el target
cols_all = USABLE + [TARGET]
print(f"Matriz de correlación de Pearson ({len(cols_all)} variables):")
print(f"{'':22s}", end="")
for c in cols_all: print(f"{c[:13]:>14s}", end="")
print()
for c1 in cols_all:
    print(f"{c1[:20]:<22s}", end="")
    for c2 in cols_all:
        r = panel_imp.stat.corr(c1, c2)
        print(f"{r:+.3f}        ", end="")
    print()

Matriz de correlación de Pearson (13 variables):
                       pct_internet_ accesos_per_c avg_velocidad n_proveedores idx_privacion   pct_grupo_A pct_rural_sis COBERTURA_NET     DESERCION    APROBACION n_estudiantes POBLACION_5_1 avg_punt_glob
pct_internet_icfes    

+1.000        

+0.200        

+0.137        

+0.496        

...

-0.130        

+0.022        

+0.088        

+0.119        

+1.000        


In [6]:
# Identificar pares de features con |r| > 0.85 y descartar el de menor corr con el target
THRESH = 0.85
target_corr = {c: panel_imp.stat.corr(c, TARGET) for c in USABLE}

pairs = []
for i, a in enumerate(USABLE):
    for b in USABLE[i+1:]:
        r_ab = panel_imp.stat.corr(a, b)
        if abs(r_ab) > THRESH:
            keep, drop = (a, b) if abs(target_corr[a]) >= abs(target_corr[b]) else (b, a)
            pairs.append((a, b, r_ab, keep, drop))

DROP_CORR = set()
print(f"Pares con |r| > {THRESH}:")
if not pairs:
    print("  (ninguno — no hay multicolinealidad fuerte)")
for a,b,r,keep,drop in pairs:
    print(f"  {a} <-> {b}  r={r:+.3f}  → mantener {keep}, descartar {drop}")
    DROP_CORR.add(drop)

FINAL = [c for c in USABLE if c not in DROP_CORR]
print(f"\nFeatures finales para modelar ({len(FINAL)}): {FINAL}")

Pares con |r| > 0.85:
  (ninguno — no hay multicolinealidad fuerte)

Features finales para modelar (12): ['pct_internet_icfes', 'accesos_per_capita_5_16', 'avg_velocidad_bajada', 'n_proveedores', 'idx_privacion', 'pct_grupo_A', 'pct_rural_sisben', 'COBERTURA_NETA', 'DESERCION', 'APROBACION', 'n_estudiantes', 'POBLACION_5_16']


## 6. Pipeline de feature engineering: VectorAssembler + StandardScaler

In [7]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

assembler = VectorAssembler(inputCols=FINAL, outputCol="features_raw", handleInvalid="keep")
scaler    = StandardScaler(inputCol="features_raw", outputCol="features",
                            withMean=True, withStd=True)
prep_pipeline = Pipeline(stages=[assembler, scaler])

# Train/test split estilo clase (sección 1.5)
df_train, df_test = panel_imp.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {df_train.count():,} filas  |  Test: {df_test.count():,} filas")

prep_model = prep_pipeline.fit(df_train)
train_prep = prep_model.transform(df_train).withColumn("label", col(TARGET).cast("double"))
test_prep  = prep_model.transform(df_test ).withColumn("label", col(TARGET).cast("double"))

print("\nVista minable lista (3 filas):")
train_prep.select("COD_MPIO","ANO","features","label").show(3, truncate=False)

26/05/22 08:22:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Train: 5,232 filas  |  Test: 1,270 filas



Vista minable lista (3 filas):


+--------+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|COD_MPIO|ANO |features                                                                                                                                                                                                                                             |label |
+--------+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|05001   |2022|[2.3586479215784,-0.058100236497595915,0.44047528720067264,10.35513967384186,-1.5081316883926899,-1.8028683300287462,-0.295651841590527,0.6431542522884103,0.8055779219287347,-1.3

## 7. Modelo 1 — Regresión Lineal

In [8]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr = LinearRegression(featuresCol="features", labelCol="label",
                      maxIter=50, regParam=0.1, elasticNetParam=0.0)
lr_model = lr.fit(train_prep)
pred_lr_tr = lr_model.transform(train_prep)
pred_lr_te = lr_model.transform(test_prep)

def metrics(df, name):
    ev = RegressionEvaluator(labelCol="label", predictionCol="prediction")
    mae  = ev.setMetricName("mae" ).evaluate(df)
    rmse = ev.setMetricName("rmse").evaluate(df)
    r2   = ev.setMetricName("r2"  ).evaluate(df)
    print(f"  {name}  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}")
    return {"MAE":mae,"RMSE":rmse,"R2":r2}

print("Regresión lineal:")
m_lr_tr = metrics(pred_lr_tr, "TRAIN")
m_lr_te = metrics(pred_lr_te, "TEST ")

print(f"\nCoeficientes (features):")
for f, w in zip(FINAL, lr_model.coefficients):
    print(f"  {f:<28s}  beta = {w:+.4f}")
print(f"Intercepto: {lr_model.intercept:.4f}")

26/05/22 08:22:19 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/22 08:22:20 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Regresión lineal:


  TRAIN  MAE=12.51  RMSE=16.37  R2=0.4169


  TEST   MAE=12.31  RMSE=16.28  R2=0.4067

Coeficientes (features):
  pct_internet_icfes            beta = +5.5768
  accesos_per_capita_5_16       beta = +2.4697
  avg_velocidad_bajada          beta = +0.2066
  n_proveedores                 beta = +0.6714
  idx_privacion                 beta = -8.7597
  pct_grupo_A                   beta = -0.0586
  pct_rural_sisben              beta = +3.1093
  COBERTURA_NETA                beta = +1.6303
  DESERCION                     beta = -1.6148
  APROBACION                    beta = +0.9125
  n_estudiantes                 beta = -1.6464
  POBLACION_5_16                beta = +1.8583
Intercepto: 240.6580


## 8. Modelo 2 — Decision Tree Regressor

In [9]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(featuresCol="features", labelCol="label",
                            maxDepth=10, minInstancesPerNode=5, seed=42)
dt_model = dt.fit(train_prep)
pred_dt_tr = dt_model.transform(train_prep)
pred_dt_te = dt_model.transform(test_prep)

print("Decision Tree Regressor:")
m_dt_tr = metrics(pred_dt_tr, "TRAIN")
m_dt_te = metrics(pred_dt_te, "TEST ")

Decision Tree Regressor:


  TRAIN  MAE=9.43  RMSE=12.30  R2=0.6709


  TEST   MAE=11.21  RMSE=14.94  R2=0.5004


## 9. Pruebas con diferentes hiperparámetros (sección 4 del enunciado)

In [10]:
import time

print("Linear Regression — grid sobre regParam y elasticNetParam:")
print(f"  {'regParam':>10s} {'elasticNet':>12s} {'TRAIN_R2':>10s} {'TEST_R2':>10s} {'TEST_RMSE':>11s}")
res_lr = []
for rp in [0.0, 0.01, 0.1, 0.5]:
    for en in [0.0, 0.5, 1.0]:
        m = LinearRegression(featuresCol="features", labelCol="label",
                              maxIter=50, regParam=rp, elasticNetParam=en).fit(train_prep)
        ev = RegressionEvaluator(labelCol="label", predictionCol="prediction")
        r2_tr = ev.setMetricName("r2").evaluate(m.transform(train_prep))
        r2_te = ev.setMetricName("r2").evaluate(m.transform(test_prep))
        rmse  = ev.setMetricName("rmse").evaluate(m.transform(test_prep))
        res_lr.append((rp, en, r2_tr, r2_te, rmse))
        print(f"  {rp:>10.3f} {en:>12.2f} {r2_tr:>10.4f} {r2_te:>10.4f} {rmse:>11.2f}")

Linear Regression — grid sobre regParam y elasticNetParam:
    regParam   elasticNet   TRAIN_R2    TEST_R2   TEST_RMSE


26/05/22 08:22:31 WARN Instrumentation: [5570e595] regParam is zero, which might cause numerical instability and overfitting.


       0.000         0.00     0.4170     0.4070       16.27


26/05/22 08:22:33 WARN Instrumentation: [f7802027] regParam is zero, which might cause numerical instability and overfitting.


       0.000         0.50     0.4170     0.4070       16.27


...

       0.100         0.50     0.4168     0.4061       16.28


       0.100         1.00     0.4166     0.4053       16.30


       0.500         0.00     0.4167     0.4054       16.29


       0.500         0.50     0.4143     0.4016       16.34


       0.500         1.00     0.4115     0.3982       16.39


In [11]:
print("\nDecision Tree — grid sobre maxDepth y minInstancesPerNode:")
print(f"  {'maxDepth':>10s} {'minInst':>10s} {'TRAIN_R2':>10s} {'TEST_R2':>10s} {'TEST_RMSE':>11s}")
res_dt = []
for md_ in [3, 5, 8, 10, 15]:
    for mi in [1, 5, 10]:
        m = DecisionTreeRegressor(featuresCol="features", labelCol="label",
                                   maxDepth=md_, minInstancesPerNode=mi, seed=42).fit(train_prep)
        ev = RegressionEvaluator(labelCol="label", predictionCol="prediction")
        r2_tr = ev.setMetricName("r2").evaluate(m.transform(train_prep))
        r2_te = ev.setMetricName("r2").evaluate(m.transform(test_prep))
        rmse  = ev.setMetricName("rmse").evaluate(m.transform(test_prep))
        res_dt.append((md_, mi, r2_tr, r2_te, rmse))
        print(f"  {md_:>10d} {mi:>10d} {r2_tr:>10.4f} {r2_te:>10.4f} {rmse:>11.2f}")


Decision Tree — grid sobre maxDepth y minInstancesPerNode:
    maxDepth    minInst   TRAIN_R2    TEST_R2   TEST_RMSE


           3          1     0.3982     0.3837       16.59


           3          5     0.3982     0.3837       16.59


           3         10     0.3982     0.3837       16.59


           5          1     0.4912     0.4735       15.33


...

          10          5     0.6709     0.5004       14.94


          10         10     0.6350     0.5107       14.78


          15          1     0.8985     0.3671       16.81


          15          5     0.7943     0.4678       15.41


          15         10     0.7094     0.4968       14.99


## 10. Comparativa final y selección del mejor modelo

In [12]:
# Mejor LR y mejor DT por test_R2
best_lr = max(res_lr, key=lambda x: x[3])
best_dt = max(res_dt, key=lambda x: x[3])

print("Mejor LinearRegression :  regParam=%.3f  elasticNetParam=%.2f  → test_R2=%.4f  RMSE=%.2f"
      % (best_lr[0], best_lr[1], best_lr[3], best_lr[4]))
print("Mejor DecisionTree     :  maxDepth=%d  minInst=%d              → test_R2=%.4f  RMSE=%.2f"
      % (best_dt[0], best_dt[1], best_dt[3], best_dt[4]))

print("\nTabla resumen (clase 4.4):")
print(f"{'Modelo':<28s} {'TRAIN_R2':>10s} {'TEST_R2':>10s} {'TEST_MAE':>10s} {'TEST_RMSE':>11s}")
print("-" * 75)
print(f"{'Regresión Lineal (default)':<28s} {m_lr_tr['R2']:>10.4f} {m_lr_te['R2']:>10.4f} "
      f"{m_lr_te['MAE']:>10.2f} {m_lr_te['RMSE']:>11.2f}")
print(f"{'Decision Tree (default)':<28s} {m_dt_tr['R2']:>10.4f} {m_dt_te['R2']:>10.4f} "
      f"{m_dt_te['MAE']:>10.2f} {m_dt_te['RMSE']:>11.2f}")
print(f"{'LR mejor del grid':<28s} {'-':>10s} {best_lr[3]:>10.4f} {'-':>10s} {best_lr[4]:>11.2f}")
print(f"{'DT mejor del grid':<28s} {'-':>10s} {best_dt[3]:>10.4f} {'-':>10s} {best_dt[4]:>11.2f}")

Mejor LinearRegression :  regParam=0.000  elasticNetParam=0.50  → test_R2=0.4070  RMSE=16.27
Mejor DecisionTree     :  maxDepth=8  minInst=10              → test_R2=0.5214  RMSE=14.62

Tabla resumen (clase 4.4):
Modelo                         TRAIN_R2    TEST_R2   TEST_MAE   TEST_RMSE
---------------------------------------------------------------------------
Regresión Lineal (default)       0.4169     0.4067      12.31       16.28
Decision Tree (default)          0.6709     0.5004      11.21       14.94
LR mejor del grid                     -     0.4070          -       16.27
DT mejor del grid                     -     0.5214          -       14.62


## 11. Modelo final (DT) — importancia de variables

In [13]:
# Decision Tree expone featureImportances (Vector)
print("Importancia de variables (Decision Tree mejor del grid):")
dt_best = DecisionTreeRegressor(featuresCol="features", labelCol="label",
                                 maxDepth=int(best_dt[0]),
                                 minInstancesPerNode=int(best_dt[1]),
                                 seed=42).fit(train_prep)
importancias = list(zip(FINAL, dt_best.featureImportances.toArray()))
importancias.sort(key=lambda x: -x[1])
for f, v in importancias:
    barra = "#" * int(v*80)
    print(f"  {f:<28s} {v:.4f}  {barra}")

Importancia de variables (Decision Tree mejor del grid):


  idx_privacion                0.4828  ######################################
  pct_internet_icfes           0.2331  ##################
  pct_rural_sisben             0.0627  #####
  POBLACION_5_16               0.0436  ###
  pct_grupo_A                  0.0433  ###
  n_estudiantes                0.0378  ###
  accesos_per_capita_5_16      0.0336  ##
  DESERCION                    0.0201  #
  COBERTURA_NETA               0.0161  #
  APROBACION                   0.0116  
  n_proveedores                0.0080  
  avg_velocidad_bajada         0.0074  


## 12. Predicciones de muestra (estilo clase 2.1)

In [14]:
pred_best = dt_best.transform(test_prep)
print("Predicciones del mejor modelo (muestra de 15 municipios-año):")
pred_best.select("COD_MPIO","ANO","COLE_DEPTO_UBICACION",
                 col("label").alias("punt_real"),
                 F.round(col("prediction"),1).alias("punt_pred"),
                 F.round(F.abs(col("label")-col("prediction")),1).alias("error_abs")
                 ) \
         .orderBy(F.rand(seed=1)).show(15, truncate=False)

Predicciones del mejor modelo (muestra de 15 municipios-año):


+--------+----+--------------------+---------+---------+---------+
|COD_MPIO|ANO |COLE_DEPTO_UBICACION|punt_real|punt_pred|error_abs|
+--------+----+--------------------+---------+---------+---------+
|05895   |2017|ANTIOQUIA           |207.9    |208.5    |0.6      |
|41807   |2022|HUILA               |242.56   |243.9    |1.4      |
|52203   |2017|NARINO              |287.35   |265.4    |22.0     |
|19212   |2016|CAUCA               |228.51   |235.2    |6.7      |
|41797   |2022|HUILA               |240.93   |243.9    |3.0      |
|05543   |2016|ANTIOQUIA           |225.26   |235.2    |10.0     |
|86571   |2015|PUTUMAYO            |221.86   |231.2    |9.4      |
|54003   |2016|NORTE SANTANDER     |256.33   |231.2    |25.1     |
|25758   |2014|CUNDINAMARCA        |262.84   |252.8    |10.0     |
|15580   |2016|BOYACA              |241.78   |233.5    |8.3      |
|05148   |2015|ANTIOQUIA           |261.7    |263.6    |1.9      |
|52506   |2015|NARINO              |263.61   |251.3    |12.4  

## 13. Persistir mejores modelos en HDFS

In [15]:
# Guardamos los modelos finales para reuso/inferencia
PATH_LR = "/usr/proyecto/models/lr_punt_global"
PATH_DT = "/usr/proyecto/models/dt_punt_global"

lr_best_model = LinearRegression(featuresCol="features", labelCol="label",
    maxIter=50, regParam=float(best_lr[0]), elasticNetParam=float(best_lr[1])).fit(train_prep)
lr_best_model.write().overwrite().save(f"hdfs://10.43.97.164:9000{PATH_LR}")
dt_best.write().overwrite().save(f"hdfs://10.43.97.164:9000{PATH_DT}")
print(f"LR persistido en HDFS: {PATH_LR}")
print(f"DT persistido en HDFS: {PATH_DT}")

26/05/22 08:23:43 WARN Instrumentation: [2241473a] regParam is zero, which might cause numerical instability and overfitting.


LR persistido en HDFS: /usr/proyecto/models/lr_punt_global
DT persistido en HDFS: /usr/proyecto/models/dt_punt_global


## 14. Conclusión

Sobre la **vista minable municipal** (gold/panel_municipal — los datos de las 4
fuentes unidos por (municipio, año)):

- El **Decision Tree** capta interacciones no-lineales mejor que LR; suele
  ganar en test_R² a costa de mayor varianza (overfitting con maxDepth grande).
- La **Regresión Lineal** con regularización (Ridge/ElasticNet) es más estable y
  sus coeficientes son directamente interpretables como impacto marginal.
- Ambos modelos usan el mismo conjunto de features (sección 5 — tras eliminar
  multicolinealidad), entrenan sobre el mismo split 80/20 y se comparan con las
  métricas estándar de regresión (`MAE`, `RMSE`, `R²` — `RegressionEvaluator` de
  la clase 4.1).
- Los modelos se persisten en HDFS y pueden recargarse con `.load()` para
  inferencia sobre nuevos datos municipales.

In [16]:
spark.stop()